# Mathematical Background for PCA

Companion notebook for Section 2 of the lecture notes.

Objectives:

- represent a dataset as a data matrix;
- compute means, variances, covariances, and covariance matrices;
- visualize covariance patterns;
- connect eigenvectors, eigenvalues, SVD, and PCA directions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


## 1. The data matrix

Rows are observations and columns are variables/features.


In [ ]:
X = np.array([
    [1.0, 2.0, 1.0],
    [2.0, 3.0, 2.0],
    [3.0, 5.0, 2.0],
    [4.0, 4.0, 4.0],
    [5.0, 6.0, 5.0],
])

pd.DataFrame(X, columns=['x1', 'x2', 'x3'])


In [ ]:
mean_vector = X.mean(axis=0)
X_centered = X - mean_vector

print('Mean vector:', mean_vector)
print('Column means after centering:', X_centered.mean(axis=0))
pd.DataFrame(X_centered, columns=['x1_centered', 'x2_centered', 'x3_centered'])


## 2. Variance and covariance

Sample variance divides by `m - 1`. Sample covariance averages products of centered values, also dividing by `m - 1`.


In [ ]:
m = X.shape[0]
manual_variances = (X_centered ** 2).sum(axis=0) / (m - 1)
numpy_variances = X.var(axis=0, ddof=1)

pd.DataFrame({'manual': manual_variances, 'numpy': numpy_variances}, index=['x1', 'x2', 'x3'])


In [ ]:
def sample_covariance(a, b):
    return ((a - a.mean()) * (b - b.mean())).sum() / (len(a) - 1)

print('cov(x1, x2):', sample_covariance(X[:, 0], X[:, 1]))
print('cov(x1, x3):', sample_covariance(X[:, 0], X[:, 2]))
print('cov(x2, x3):', sample_covariance(X[:, 1], X[:, 2]))


## 3. Covariance matrix

For centered data, `Sigma = X_centered.T @ X_centered / (m - 1)`. The diagonal entries are variances; off-diagonal entries are covariances.


In [ ]:
Sigma_manual = X_centered.T @ X_centered / (m - 1)
Sigma_numpy = np.cov(X, rowvar=False)

print('Manual covariance matrix:\n', Sigma_manual)
print('\nnp.cov covariance matrix:\n', Sigma_numpy)
print('\nSymmetric?', np.allclose(Sigma_manual, Sigma_manual.T))


## 4. Visual interpretation of covariance

Positive covariance, near-zero covariance, and negative covariance produce different point-cloud shapes.


In [ ]:
rng = np.random.default_rng(7)
n = 200
x = rng.normal(size=n)
examples = {
    'positive': (x, 1.5 * x + rng.normal(scale=0.5, size=n)),
    'near zero': (x, rng.normal(size=n)),
    'negative': (x, -1.5 * x + rng.normal(scale=0.5, size=n)),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True, sharey=True)
for ax, (title, (a, b)) in zip(axes, examples.items()):
    ax.scatter(a, b, s=15, alpha=0.7)
    ax.set_title(f'{title}\ncov={sample_covariance(a, b):.2f}')
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
plt.tight_layout()
plt.show()


## 5. Dot product and orthogonality

Orthogonal vectors have dot product zero. PCA components form an orthonormal basis for the retained subspace.


In [ ]:
u = np.array([1.0, 0.0])
v = np.array([0.0, 1.0])
w = np.array([1.0, 1.0]) / np.sqrt(2)

for name, vec in [('v', v), ('w', w)]:
    print(f'u dot {name} =', np.dot(u, vec))
    print(f'angle with {name} =', np.degrees(np.arccos(np.dot(u, vec) / (np.linalg.norm(u) * np.linalg.norm(vec)))))


## 6. Eigenvectors of a covariance matrix

Eigenvectors are directions that are scaled but not rotated by the matrix. For covariance matrices, they identify principal directions of variation.


In [ ]:
A = np.array([[9.0, 5.0],
              [5.0, 4.0]])

eigvals, eigvecs = np.linalg.eigh(A)
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

print('Eigenvalues:', eigvals)
print('Eigenvectors as columns:\n', eigvecs)
print('A @ first eigenvector:', A @ eigvecs[:, 0])
print('lambda * first eigenvector:', eigvals[0] * eigvecs[:, 0])


In [ ]:
rng = np.random.default_rng(3)
cloud = rng.multivariate_normal(mean=[0, 0], cov=A, size=400)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(cloud[:, 0], cloud[:, 1], alpha=0.35, s=12)
origin = np.zeros(2)
for j in range(2):
    direction = eigvecs[:, j] * np.sqrt(eigvals[j])
    ax.arrow(*origin, *direction, width=0.05, color=f'C{j+1}', length_includes_head=True)
    ax.text(*(direction * 1.15), f'PC{j+1}')
ax.set_aspect('equal', adjustable='box')
ax.set_title('Eigenvectors identify directions of variation')
plt.show()


## 7. SVD connection

If `X_centered = U S V.T`, the columns of `V` are the PCA directions. The covariance eigenvalues are `S ** 2 / (m - 1)`.


In [ ]:
X2 = cloud - cloud.mean(axis=0)
U, S, Vt = np.linalg.svd(X2, full_matrices=False)
svd_directions = Vt.T
svd_eigenvalues = S ** 2 / (X2.shape[0] - 1)

cov_eigenvalues, cov_eigenvectors = np.linalg.eigh(np.cov(X2, rowvar=False))
order = np.argsort(cov_eigenvalues)[::-1]
cov_eigenvalues = cov_eigenvalues[order]
cov_eigenvectors = cov_eigenvectors[:, order]

print('Eigenvalues from covariance:', cov_eigenvalues)
print('Eigenvalues from SVD:', svd_eigenvalues)
print('\nAbsolute agreement between directions:')
print(np.abs(cov_eigenvectors.T @ svd_directions))


## Takeaways

- PCA depends on centered numeric data.
- The covariance matrix summarizes feature spread and pairwise linear relationships.
- PCA directions can be obtained either from covariance eigenvectors or from SVD of the centered data matrix.
